# Triple Bottom：用 qust 标注三重底确认

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


来源参考：[Investopedia](https://www.investopedia.com/terms/t/triplebottom.asp)

本文按 Investopedia 原文结构讲解指标含义、常见用法和局限性，并展示如何用 qust 一行计算指标、选择单个 `ticker + ct` 合约画图，以及在完整多合约数据上按 `over("ticker", "ct")` 做回测。


## 1. Investopedia 原文内容完整改写：Triple Bottom

### 什么是 Triple Bottom
Triple Bottom 是一种看涨反转形态，由三个相近低点组成。价格三次下跌到同一区域附近都没有继续破位，说明该区域存在较强支撑。图形上类似三个低点排列在同一水平附近，随后价格向上突破阻力。

### 形态确认
三个低点本身还不够。低点之间的反弹会形成上方阻力，只有价格突破这个阻力区域，Triple Bottom 才被认为确认。否则价格仍可能只是横盘震荡，甚至第四次跌破支撑。

### 形态结构
典型结构包括：第一次下跌触底反弹；第二次回落到相近区域再反弹；第三次再次测试支撑并守住；最后向上突破前面反弹高点形成的阻力。三个低点不需要完全相等，但应足够接近。

### 市场心理
每次价格跌到同一区域都有买盘承接，说明卖方无法继续压低价格。反复测试后，如果价格转而突破阻力，说明买方不仅能防守，还开始掌握上攻能力。

### 使用方式
交易者通常等待突破确认后再考虑看多，风险可能放在第三个低点或支撑区下方。目标价有时用支撑到阻力的高度向上估计。成交量在突破时增加会增强解释。

### 局限性
Triple Bottom 形成时间可能较长，识别有滞后。低点相近程度、低点间隔、阻力定义都存在参数选择。很多看起来像三重底的结构最终可能只是震荡区间，未突破前不能当作完成形态。

## 2. 从文章到 qust 算子的落地

qust 检测三个相近 pivot low，计算中间反弹阻力，并只在 close 突破阻力时输出 `triple_bottom=True`。这对应文章强调的 breakout confirmation。

## 3. qust 一行调用

```python
col("high", "low", "close").investopedia.triple_bottom()
```

输入列顺序：`high, low, close`。

输出列：`triple_bottom`, `triple_bottom_level`, `triple_bottom_resistance`, `triple_bottom_breakout`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实GitHub K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("high", "low", "close").investopedia.triple_bottom()
triple_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
plot_data = (
    triple_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    col("triple_bottom").cast(pl.UInt32).sum().alias("triple_bottom_count"),
).calc_data(triple_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 12)


triple_bottom_count
u32
12831


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 Jupyter 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
triple_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("triple_price", show_axis_label=True)
        .kline(),
    col("datetime", "triple_bottom_level", "triple_bottom_resistance")
        .monitor("triple_price", show_axis_label=True)
        .line(),
    col("datetime", "close", "triple_bottom")
        .monitor("triple_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_up, color="#50fa7b", width=0.45),
).monitor.make_monitor("black").monitor.add_grid([
    ["triple_price"],
]).runtime()

triple_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Triple Bottom 策略回测

三重底是看涨反转结构。策略把 `triple_bottom` 突破确认作为多头入场来源，下一根 K 线执行，3% 止盈、1.5% 止损，持仓除以 `col.all.fp.vol_pms()` 后再进入 `bt.price()`。

In [4]:
TAKE_PROFIT = 0.03
STOP_LOSS = 0.015

indicator_cols = col("high", "low", "close").investopedia.triple_bottom()
strategy_daily_expr = (
    col
    .with_cols(indicator_cols)
    .with_cols(
        (col("triple_bottom")).fill_null(col.lit(False)).alias("open_long_raw"),
        (col.lit(False)).fill_null(col.lit(False)).alias("open_short_raw"),
    )
    # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
    .with_cols(
        col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
        col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
        col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
        col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
        col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
    )
    .with_cols(
        (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
            .fill_null(col.lit(False))
            .alias("exit_long_sig"),
        (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
            .fill_null(col.lit(False))
            .alias("exit_short_sig"),
    )
    .with_cols(
        col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
            .stra.to_hold_two_sides()
            .expanding()
            .alias("hold")
    )
    .with_cols((col("hold") / col.all.fp.vol_pms()).alias("hold"))
    .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
    .over("ticker", "ct")
    .select(
        col("pnl")
            .sum()
            .group_by(col("datetime").dt.date().alias("date"))
            .batch.sort("date")
            .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
            .select("date", "pnl", "pnl_cum")
    )
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = col("date", "pnl").bt.returns_stats(periods_per_year=252).calc_data(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


metric,value,value_float
str,str,f64
"""Start Index""","""2022-01-04""",null
"""End Index""","""2024-12-31""",null
"""Total Duration""","""1092 days, 0:00:00""",null
"""Total Return [%]""","""-3.502225367877762e+73""",-3.5022e73
"""Benchmark Return [%]""",null,null
"""Annualized Return [%]""",null,null
"""Annualized Volatility [%]""","""3367.230818117706""",3367.230818
"""Max Drawdown [%]""","""2154116.4036795683""",2.1541e6
…,…,…


In [5]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,-0.771912,27.526395
2024-12-19,-2.838989,24.687406
2024-12-20,-1.101641,23.585765
2024-12-21,-0.125345,23.460419
2024-12-23,-0.231312,23.229108
2024-12-24,1.712315,24.941422
2024-12-25,-1.441741,23.499682
2024-12-26,-0.240253,23.259429
2024-12-27,-2.392497,20.866931


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。